# Module 1: Introduction to Amazon Bedrock AgentCore

---

## What is Amazon Bedrock AgentCore?

**Amazon Bedrock AgentCore** is a suite of managed services purpose-built for deploying, securing, and operating AI agents in production. It removes the undifferentiated heavy lifting of agent infrastructure so you can focus on building agent logic.

### Why AgentCore matters

Building a prototype AI agent is easy. Getting one into production is hard. You need to solve for:

- **Hosting & scaling** -- Where does the agent run? How does it scale to thousands of users?
- **Tool access** -- How does the agent execute code, browse the web, or call APIs safely?
- **Memory** -- How does the agent remember past conversations and user preferences?
- **Authentication** -- How do you know who is talking to the agent?
- **Authorization** -- What is each user allowed to do?
- **Observability** -- How do you debug failures and monitor quality?
- **Evaluation** -- How do you know the agent is actually good?

AgentCore provides a managed answer for every one of these challenges.

---

## The 9 AgentCore Services

AgentCore is composed of 9 services that work together to form a complete agent platform:

### Compute & Tools

| Service | What it does |
|---------|-------------|
| **Runtime** | Hosts and scales your agent in a secure, serverless environment. Each session gets its own dedicated microVM with isolated CPU, memory, and filesystem -- so your agent code only handles one session at a time and conversation history is preserved automatically. |
| **Code Interpreter** | Gives your agent a sandboxed Python execution environment. The agent can write and run code, generate charts, and process files -- all in a secure sandbox. |
| **Browser Tool** | Gives your agent the ability to browse the web. It can navigate pages, extract content, and interact with web applications. |

### Memory & State

| Service | What it does |
|---------|-------------|
| **Memory** | Provides persistent, cross-session memory. The agent remembers past conversations, user preferences, and extracted facts without you managing a database. |

### Security & Access

| Service | What it does |
|---------|-------------|
| **Gateway** | A managed MCP gateway that converts your APIs, Lambda functions, and existing services into MCP-compatible tools -- and connects to pre-existing MCP servers -- making them available to AI agents through a secure endpoint. |
| **Identity** | Integrates with identity providers (like Amazon Cognito) to authenticate end users before they reach your agent. |
| **Policy** | Enforces fine-grained authorization using [Cedar](https://www.cedarpolicy.com/) policies. Control what each user or group can do -- down to individual tool calls. |

### Quality & Operations

| Service | What it does |
|---------|-------------|
| **Observability** | Emits structured traces, metrics, and logs for every agent invocation. Integrates with Amazon CloudWatch for dashboards and alarms. |
| **Evaluations** | Runs automated quality assessments against your agent using configurable test suites. Validates that your agent meets accuracy, safety, and latency targets. |

### How they fit together

![AgentCore Overview](../shared/img/ac_overviewchart.svg)

---

## The Strands Agents SDK

AgentCore is **framework-agnostic** -- you can deploy agents built with any framework. However, in this workshop we use the **[Strands Agents SDK](https://strandsagents.com)**, an open-source Python SDK from AWS that is designed to work seamlessly with AgentCore.

Strands provides:

- A simple `Agent` class with built-in tool calling and multi-turn conversation
- Native integration with AgentCore services (Memory, Code Interpreter, Browser Tool)
- Automatic trace emission for AgentCore Observability
- A lightweight, composable architecture

---

## The AgentCore CLI (FYI)

AgentCore provides a CLI for creating, developing, and deploying agents. The CLI is an npm package (`@aws/agentcore`) that requires **Node.js 20.x or later** and **uv** (for Python agents).

> **Note:** We won't use the CLI in this workshop -- we'll work directly with boto3 so you can see the underlying API calls. But the CLI is useful to know about for when you want to move fast outside of a workshop setting.

### Installing the CLI

On your local machine, you would install it globally via npm:

```bash
npm install -g @aws/agentcore
```

### CLI help output

Running `agentcore --help` shows the available commands:

```
Usage: agentcore [options] [command]

Amazon Bedrock AgentCore CLI

Options:
  -V, --version       output the version number
  -h, --help          display help for command

Commands:
  create [options]    Create a new AgentCore project
  dev [options]       Start a local development server
  deploy [options]    Deploy your agent infrastructure to AWS
  invoke [options]    Invoke your deployed agent
  add [options]       Add resources to your project
  remove [options]    Remove resources from your project
  help [command]      display help for command
```

### Key CLI commands

The CLI organizes around a simple project lifecycle:

| Command | Description |
|---------|-------------|
| `agentcore create` | Create a new AgentCore project (a wizard guides you through agent setup) |
| `agentcore dev` | Start a local development server to test your agent |
| `agentcore deploy` | Deploy your agent infrastructure to AWS |
| `agentcore invoke` | Invoke your deployed agent |
| `agentcore add` | Add resources (agents, memory, identity, evaluators, targets) |
| `agentcore remove` | Remove resources from the project |

### Example: Creating a project

Running `agentcore create` walks you through an interactive wizard:

```
$ agentcore create

? Project name: my-agent
? Select agent framework: Strands Agents
? Select model: Claude Sonnet
? Add tools? Yes
? Select tools: Code Interpreter, Memory

Creating project my-agent...
✔ Project created successfully!

Next steps:
  cd my-agent
  agentcore dev     # start local development
  agentcore deploy  # deploy to AWS
```

Since this workshop focuses on understanding the underlying APIs, we use boto3 directly in the modules that follow. This gives you full visibility into the API calls, parameters, and response structures that the CLI abstracts away.

---

## AWS SDKs and the Workshop Approach

AgentCore resources can be managed through three interfaces:

1. **AgentCore CLI** (`agentcore`) — Quick commands for listing, deploying, and testing from the terminal
2. **AWS CDK** — Infrastructure as Code for production deployments
3. **AWS SDKs** (e.g., `boto3` for Python) — Programmatic access to all AgentCore APIs

In this workshop, we primarily use **boto3** because:
- You see the exact API calls that create and manage AgentCore resources
- You understand the parameters, configurations, and response structures
- The same calls work in your production code

### Helper functions

We provide a `shared/` utilities directory with helper functions for:
- **`ensure_ready(module)`** — Smart catch-up script. Checks what exists, creates what's missing. Run this at the start of any module to guarantee prerequisites are in place (even if you skipped modules or broke something).
- **`deploy_agent.deploy()`** — Packages agent code into a zip, uploads to S3, and creates/updates the Runtime. We hide this because the packaging boilerplate isn't the learning objective.
- **`progress.show()`** — Visual progress tracker.

**What we DON'T hide:** The actual AgentCore API calls for creating resources (Memory, Gateway, Policy, Evaluators) and invoking agents. These are the concepts you need to understand, so you'll see them as raw boto3 calls.

In [ ]:
# Example: boto3 clients for AgentCore
import boto3

# Control plane -- create and manage resources (Runtime, Memory, Gateway, Policy)
control_client = boto3.client("bedrock-agentcore-control", region_name="us-east-1")

# Data plane -- invoke agents, interact with running services
data_client = boto3.client("bedrock-agentcore", region_name="us-east-1")

print("Control plane actions:", [a for a in dir(control_client) if a.startswith("create_")][:5])
print("Data plane actions:", [a for a in dir(data_client) if a.startswith("invoke")][:5])

---

## The Journey Ahead

Here is what you will build in each remaining module:

### Module 2: Deploy Your First Agent to Runtime
Build Aria as a Strands agent and deploy it to AgentCore Runtime. By the end of this module, Aria will be running as a managed, scalable endpoint that you can invoke via the CLI and SDK.

### Module 3: Add Code Interpreter & Browser Tools
Give Aria superpowers. With Code Interpreter, she can write and execute Python code, generate visualizations, and process files. With Browser Tool, she can navigate the web to research topics and extract information.

### Module 4: Add Persistent Memory
Enable Aria to remember. Using AgentCore Memory, Aria will retain context across sessions -- remembering user preferences, past conversations, and extracted facts.

### Module 5: Connect Gateway & Identity
Give Aria access to external APIs. AgentCore Gateway is a managed MCP gateway that converts the Task Management REST API into MCP-compatible tools the agent can call. Identity integration with Amazon Cognito ensures every user's JWT flows through the stack so their data stays private.

### Module 6: Enforce Cedar Policies
Add fine-grained authorization. Using Cedar policies through AgentCore Policy, you will control exactly what each user is allowed to do -- down to individual tool calls and data access.

### Module 7: Observability & Evaluations
Make Aria observable and testable. AgentCore Observability gives you structured traces and metrics for every invocation. Evaluations let you run automated quality checks to ensure Aria meets your standards.

### Module 8: Full Production Deployment
Bring it all together. Deploy the complete, production-ready version of Aria with all 9 AgentCore services wired up and working in concert.

---

## Key Documentation Links

Keep these bookmarked -- you will reference them throughout the workshop:

- **Amazon Bedrock AgentCore**: [https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- **AgentCore Runtime**: [https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)
- **Strands Agents SDK**: [https://strandsagents.com](https://strandsagents.com)
- **Cedar Policy Language**: [https://www.cedarpolicy.com](https://www.cedarpolicy.com)

---

## Mark Module Complete

In [ ]:
import sys
sys.path.insert(0, '..')

from shared.progress import show

show("01")

---

**Next up: [Module 2 -- Deploy Your First Agent to Runtime](../02-runtime/notebook.ipynb)**